<div style="background-color:#B5121B;color:#ffffff;padding:16px 22px;border-radius:6px;font-family:Arial, Helvetica, sans-serif;">
<div style="font-size:12px;letter-spacing:0.08em;text-transform:uppercase;opacity:0.85;">PHYS7101 &middot; Numerical Methods and Data Analysis</div>
<div style="font-size:21px;font-weight:700;margin-top:6px;line-height:1.3;">Week 3 — Interpolation and Extrapolation</div>
</div>

#### Estimating between and beyond the data: filling gaps in the Mauna Loa CO₂ record, Runge's phenomenon, and what happens when you extrapolate.
***

## Where this fits

In Week 2 you fitted a model to the tide: a smooth function with a few parameters, chosen to describe *all* the data. Interpolation is different: it passes a curve *exactly through* known points to estimate values *between* them. Extrapolation uses either one to estimate values *beyond* the data, and that is where things go wrong.

The dataset is the Mauna Loa CO₂ record, monthly since 1958. Lecture 4 covers the methods. Today you will:

1. load and clean the record;
2. fill in deliberately removed months with linear, spline and polynomial interpolation;
3. see why more points can make a polynomial *worse* (Runge's phenomenon);
4. fit candidate models to 1958–1999 (the code is provided);
5. use them to predict 2000–2026, and check against what actually happened.

**Nothing here is assessed.** Write your predictions in your logbook before you run each comparison.

## How to use this notebook today

**Core (about 1¼ hours):** Sections 0, 1(a)–(b) and 2, then run Section 3 and do 4(a) and 4(c). Parts marked *Optional* can wait.

**Working on Exercise 1?** From this notebook you need the cubic spline and the leave-out check in Section 1. Do those, then use the rest of the lab for the exercise.

**Behind on earlier weeks?** Catch up first on what Exercise 1 builds on: `week1_euler_rk4_lab.ipynb` (Sections 1–3 and 5), `week2_root_finding.ipynb` (Sections 1, 2 and 5), `week2_fitting.ipynb` (Sections 2–4) and `week2_code_and_plots.ipynb` (Section 6). This notebook will still be here.

## <span style="color:#B5121B">0. Load and clean the data</span>

The file is NOAA's monthly record, unchanged. Open it in a text editor first (or look at it on GitHub): it starts with comment lines beginning `#`, then the columns:

| Column | Meaning |
|---|---|
| `year`, `month` | calendar month |
| `decimal date` | mid-month as a decimal year |
| `average` | monthly mean CO₂ (ppm, parts per million) |
| `deseasonalized` | the same with the average seasonal cycle removed |
| `ndays` | number of days with data |
| `sdev`, `unc` | standard deviation of the daily values, and uncertainty of the monthly mean (ppm) |

The header notes say that **missing values are marked with negative numbers**, and that missing months have already been filled by NOAA.

In [ ]:
import os

# Data: read from a local datasets/ folder if you've cloned/downloaded the
# repository, otherwise straight from GitHub (e.g. in Colab). Same file either way.
DATA_URL = "https://raw.githubusercontent.com/MatSmithAstro/phys7101_2026_resources/main/workshops/week3/datasets/"

def data_path(filename):
    local = os.path.join("datasets", filename)
    return local if os.path.exists(local) else DATA_URL + filename

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

co2 = pd.read_csv(data_path("co2_mm_mlo.csv"), comment="#")   # comment="#" skips the header notes
co2.head()

In [ ]:
# TODO: how many rows have a negative ndays, sdev or unc? In which years?
# TODO: replace the negative sdev and unc values with np.nan
# TODO: plot average and deseasonalized against decimal date

In [ ]:
# Used throughout: time and CO2 as NumPy arrays
t = co2["decimal date"].to_numpy()
y = co2["average"].to_numpy()

In your logbook: describe the two features of the curve in one line each. Roughly how big is the seasonal cycle (peak to trough, ppm)?

## <span style="color:#B5121B">1. Fill the gaps</span>

Take the four years 2018–2021 and **keep only every second month**. The months you remove are the test set: you know their true values, so you can measure how well each method recovers them. This is the check from Lecture 4: leave out points, interpolate from the rest, and compare.

The monthly means have uncertainties of 0.1–0.5 ppm, much smaller than the seasonal swing, so treating them as exact values is reasonable here. (Lecture 4: interpolate exact values; fit noisy ones. Extension B shows the difference.)

**(a)** Write your own `linear_interp(x_known, y_known, x)`: for each `x`, find the two known points either side and join them with a straight line. Check it against `np.interp`.

In [ ]:
window = (t >= 2018) & (t < 2022)
t_win, y_win = t[window], y[window]

keep = np.arange(0, len(t_win), 2)            # every second month
t_known, y_known = t_win[keep], y_win[keep]

# test points: the removed months that lie inside the known range
inside = (t_win > t_known[0]) & (t_win < t_known[-1])
test = np.setdiff1d(np.where(inside)[0], keep)
t_test, y_test = t_win[test], y_win[test]
print(f"{len(t_known)} known points, {len(t_test)} test points")

In [ ]:
def linear_interp(x_known, y_known, x):
    """Piecewise-linear interpolation. x_known must be increasing.

    Returns an array of interpolated values at each x.
    """
    # TODO: for each value of x:
    #   find i such that x_known[i] <= x <= x_known[i+1]   (np.searchsorted helps)
    #   return y_known[i] + slope * (x - x_known[i])
    raise NotImplementedError

In [ ]:
try:
    mine = linear_interp(t_known, y_known, t_test)
    print("max difference from np.interp:", np.max(np.abs(mine - np.interp(t_test, t_known, y_known))))
except NotImplementedError:
    print("Not implemented yet: complete the function above, then re-run this cell.")

**(b)** Now compare three methods on the test months. **Predict first** in your logbook: which will do best?

- linear: `np.interp`;
- cubic spline: `scipy.interpolate.CubicSpline`, piecewise cubics joined smoothly (continuous first and second derivatives);
- one polynomial through *all* the known points (degree $n - 1$): `np.polynomial.Polynomial.fit(t_known, y_known, deg=len(t_known) - 1)`.

For each, compute the rms error on the test months, and plot all three on a fine grid of times with the known and test points marked.

In [ ]:
from scipy.interpolate import CubicSpline
from numpy.polynomial import Polynomial

# TODO: build the three interpolants from (t_known, y_known)
# TODO: rms error of each on (t_test, y_test)
# TODO: plot each on t_fine = np.linspace(t_known[0], t_known[-1], 1000),
#       with the known and test points; set a sensible y-range (e.g. 400-425 ppm)

**(c) Optional.** Repeat with only **every sixth month** kept (`keep = np.arange(0, len(t_win), 6)`). What happens to the linear and spline errors? Look at *which* months are left. In your logbook: why can no method recover the seasonal cycle now?

In [ ]:
# TODO: repeat (b) with every sixth month

## <span style="color:#B5121B">2. Runge's phenomenon</span>

Section 1 showed a high-degree polynomial misbehaving near the ends. Here is the classic clean example. Interpolate

$$f(x) = \frac{1}{1 + 25x^2}, \qquad -1 \le x \le 1,$$

with a single polynomial through $n$ points. The **Lagrange form** of the interpolating polynomial is

$$p(x) = \sum_{j=0}^{n-1} y_j\, L_j(x), \qquad L_j(x) = \prod_{m \ne j} \frac{x - x_m}{x_j - x_m}.$$

Each $L_j$ is 1 at $x_j$ and 0 at every other node, so $p$ passes through every point.

**(a)** Write `lagrange(x_nodes, y_nodes, x)`.

In [ ]:
def lagrange(x_nodes, y_nodes, x):
    """Evaluate the polynomial through (x_nodes, y_nodes) at x, using the Lagrange form."""
    # TODO: total = zeros like x
    # TODO: for each j: build L_j(x) as a product over m != j, then add y_nodes[j] * L_j
    raise NotImplementedError

In [ ]:
def runge(x):
    return 1 / (1 + 25 * x**2)

try:
    xn = np.array([-1.0, 0.0, 2.0])
    print(lagrange(xn, xn**2, np.array([0.5, 1.5])), "should be [0.25 2.25]")   # a parabola is reproduced exactly
except NotImplementedError:
    print("Not implemented yet: complete the function above, then re-run this cell.")

**(b)** **Predict first:** with more equally spaced points, does the interpolant get better or worse?

For $n = 5, 9, 13, 17, 21$ equally spaced nodes, plot the interpolant and print the maximum error $\max|p(x) - f(x)|$ on a fine grid.

**(c)** Repeat with **Chebyshev nodes**, $x_k = \cos\left(\frac{(2k+1)\pi}{2n}\right)$, $k = 0, \dots, n-1$, which bunch up towards the ends. Then try a cubic spline through the equally spaced nodes. Make a table of the maximum error for all three.

In [ ]:
x_fine = np.linspace(-1, 1, 2001)

# TODO: for n in [5, 9, 13, 17, 21]:
#     equally spaced nodes; Chebyshev nodes
#     max error of: lagrange (equal), lagrange (Chebyshev), CubicSpline (equal)
# TODO: plot the equally spaced interpolants for a few n, with f(x)

In your logbook: was your prediction right? Which method would you use to interpolate measured data, where you can't choose the nodes?

## <span style="color:#B5121B">3. Candidate models for the CO₂ record</span>

Interpolation can't take you beyond the data. For that you need a model. Here we fit candidate models to **1958–1999 only** (the *training* data), so that Section 4 can test them against 2000–2026.

Each model is a polynomial trend plus an annual cycle and its first harmonic:

$$y(t) = \sum_{k=0}^{d} c_k (t - 1980)^k + \sum_{h=1}^{2} \left[a_h \cos(2\pi h t) + b_h \sin(2\pi h t)\right].$$

It is linear in the parameters, so this is the design-matrix least-squares fit from Week 2 (`week2_fitting.ipynb`, Section 1). Centring on 1980 keeps the columns from becoming enormous ($2000^5$ is about $3\times10^{16}$).

**The code is provided**, because the fitting is Week 2 material: run the next three cells, and look at the rms residuals and the residual plots.

In [ ]:
def design_matrix(t, degree, n_harmonics=2):
    """Columns: (t-1980)^0 ... (t-1980)^degree, then cos and sin of 2*pi*h*t for h = 1..n_harmonics."""
    columns = [(t - 1980) ** k for k in range(degree + 1)]
    for h in range(1, n_harmonics + 1):
        columns += [np.cos(2 * np.pi * h * t), np.sin(2 * np.pi * h * t)]
    return np.column_stack(columns)

In [ ]:
try:
    print(design_matrix(np.array([1980.0, 1981.5]), degree=2).round(3))
except NotImplementedError:
    print("Not implemented yet: complete the function above, then re-run this cell.")

The cell below fits degrees $d = 1, 2, 3, 5$ to the training data (`np.linalg.lstsq(A, y, rcond=None)`), prints the rms of the residuals and plots them against time. In your logbook: which degree would you choose, on this evidence alone?

(The SIO-era months have no uncertainty, so this is an unweighted fit: rms residual plays the role χ² did in Week 2.)

In [ ]:
train = t < 2000
degrees = [1, 2, 3, 5]
params = {}

fig, axes = plt.subplots(len(degrees), 1, figsize=(9, 8), sharex=True, sharey=True)
for d, ax in zip(degrees, axes):
    A = design_matrix(t[train], d)
    theta, *_ = np.linalg.lstsq(A, y[train], rcond=None)
    params[d] = theta
    resid = y[train] - A @ theta
    print(f"degree {d}: rms residual = {np.sqrt(np.mean(resid**2)):.3f} ppm")
    ax.plot(t[train], resid, ".", ms=2, color="#B5121B")
    ax.axhline(0, color="grey", lw=0.8)
    ax.set_ylabel(f"d = {d}\nresid (ppm)")
axes[-1].set_xlabel("Year")
axes[0].set_title("Residuals, training data 1958-1999")
plt.show()

**Optional aside.** Every extra parameter lowers the residuals a little, whether or not it describes anything real. A common way to penalise extra parameters is the *Bayesian information criterion*, $\text{BIC} = \chi^2 + k\ln n$ for $k$ parameters and $n$ points. Lower is better. With an assumed $\sigma = 0.3$ ppm, compute BIC for each degree. Does it stop the quintic winning? (Model comparison returns properly in Week 9.)

In [ ]:
# Optional: BIC for each degree

## <span style="color:#B5121B">4. Extrapolate: predict 2000–2026</span>

**Predict first**, in your logbook: which of the four models will best predict the CO₂ level in 2026? Write down a number for each if you can.

**(a)** Evaluate each fitted model from 1958 to 2026. Plot the predictions with the full record, marking where the training data ends. Print each model's prediction for the last month in the file, and its rms error over 2000–2026.

In [ ]:
# TODO: for each degree: prediction = design_matrix(t, d) @ params[d]
#       print prediction[-1] (with the true y[-1]) and the rms error over t >= 2000
#       plot all predictions with the data; ax.axvline(2000) marks the end of training

**(b) Optional.** Is the quadratic *always* the right choice? Move the end of the training data to **1990** and repeat (a). Which model wins now?

In [ ]:
# TODO: refit with t < 1990 and repeat the comparison

**(c)** Two traps when extrapolating with interpolation tools. Run the cell below. What does `np.interp` return outside the data, and does it warn you? What does a cubic spline built on 1958–1999 predict for 2010?

In [ ]:
print("np.interp at 2030:", np.interp(2030, t, y))
print("np.interp at 1900:", np.interp(1900, t, y))
spline_train = CubicSpline(t[train], y[train])
print(f"cubic spline (1958-1999) at 2010: {spline_train(2010):.0f} ppm")

**(d) Optional.** Use each of the four models (trained to 1999) to predict CO₂ in **2050**. How large is the spread? In your logbook: if someone gave you only one of these numbers, what would you want to know before believing it?

In [ ]:
# TODO: predictions for 2050

## <span style="color:#FD9029">Extension A (optional): a physically motivated model</span>

Polynomials have no physical meaning. A simple physical picture: CO₂ is a pre-industrial baseline of about 280 ppm plus an excess that grows exponentially,

$$y(t) = 280 + A\,e^{(t - 1990)/\tau} + \text{annual cycle}.$$

It's nonlinear in $\tau$, so use `curve_fit` (Week 2, `week2_fitting.ipynb` Section 4). Fit 1958–1999, report $A$ and $\tau$, and predict 2026. Does building in physics help the extrapolation? What might the model be missing?

In [ ]:
from scipy.optimize import curve_fit

def co2_exp(t, A, tau, a1, b1, a2, b2):
    # TODO
    raise NotImplementedError

# TODO: fit to t < 2000 with p0 = [70, 50, 0, 0, 0, 0]; predict 2026

## <span style="color:#FD9029">Extension B (optional): interpolation vs smoothing</span>

Interpolation goes exactly through every point, noise included. For noisy data that is usually not what you want.

Take $f(x) = \sin(2\pi x)$ at 60 equally spaced points on $[0, 2]$ and add Gaussian noise with $\sigma = 0.2$ (`np.random.default_rng(2)`). Compare:

- a `CubicSpline` through the noisy points;
- `scipy.interpolate.UnivariateSpline(x, y_noisy, s=...)`, a *smoothing* spline, which may miss the points by a total squared distance `s`. Start with $s = n\sigma^2$.

Measure each one's rms difference from the true $f(x)$ on a fine grid. Then scan `s` from $0.1\,n\sigma^2$ to $10\,n\sigma^2$ and plot the rms error against `s`. What happens at each end?

In [ ]:
from scipy.interpolate import UnivariateSpline
# TODO

## Summary

- **Interpolation** estimates between known points. Linear is safe; cubic splines follow curvature; a single high-degree polynomial is not safe.
- **Runge's phenomenon:** more equally spaced points can make a polynomial interpolant worse. Chebyshev nodes or splines fix it.
- **Sampling matters:** if the samples don't resolve a variation, no method can recover it.
- **Extrapolation** depends on the model, not just the data. The best in-sample fit can be the worst predictor, and the "best" predictor changes with the training window.
- Check what your tools do at the edges: `np.interp` clamps silently; splines extrapolate wildly.

In your logbook: one sentence on when you would trust an extrapolation, and when you wouldn't.

<hr style="border:none;border-top:1px solid #BEC0C2;margin-top:32px;">
<div style="color:#555656;font-size:12px;margin-top:8px;font-family:Arial, Helvetica, sans-serif;">
PHYS7101 &mdash; Numerical Methods and Data Analysis
</div>